## Instructions {-}

1. The assignment is worth **100 points**. **See Canvas for the due date.** There is an optional bonus question worth 10 points, so you can score a maximum of 110 (out of 100) points.
2. You may discuss approaches with classmates, but write your own code and explanations. Put Python computations in Code cells and write the final answer to each question in a Markdown cell.
3. [Download the Assignment F project kit](downloads/assignment-f-kit.zip). Extract it and move the extracted `assignment-f` folder into `stat303-1/assignments/`, next to your earlier assignment folders. Check that you did not create a nested `assignment-f/assignment-f/` folder. Open `stat303-1` in VS Code, open `assignments/assignment-f/assignment_f.ipynb`, and select the course `.venv` as its kernel. pandas, seaborn, SciPy, and scikit-learn are already installed there from [Data Cleaning](Data_cleaning.ipynb).
4. **This page contains the complete assignment instructions.** The starter notebook contains numbered answer spaces and supplied code. Keep those sections in order, answer every numbered part below, and add your name to the `author` line at the top of the notebook.
5. The data files are in the kit's `data/` folder. Read them with relative path strings such as `"data/Housing_complete.csv"`; the notebook runs from the `assignment-f` folder, so these paths work without changes.
6. Do not use a `for` loop outside the bonus question. Every question in Sections 1–3 can be answered with the pandas, NumPy, and seaborn tools from the chapters.
7. Submit **one file, `assignment_f.html`**, to the **Assignment F submission in Canvas**. *Prepare and Submit the HTML* at the end of this page explains how to render it.

8. **Three of the 100 points are for properly formatting the assignment.** The breakdown is as follows:
   - Must be an HTML file rendered using Quarto (1 pt).
   - There aren't excessively long outputs of extraneous information (e.g. no printouts of entire data frames without good reason, there aren't long sections of commented-out code, etc.) (1 pt).
   - Final answers of each question are written in Markdown cells, and there is no unnecessary / redundant code or text (1 pt).

9. **Generative AI.** You may use generative AI tools (ChatGPT, Claude, GitHub Copilot, and similar) to understand concepts, interpret error messages, and debug code you have written. You may not use them to produce the code or written answers you submit. The test: you should be able to explain every line you submit and reproduce it without assistance. Disclose any use in the AI Use Disclosure section (Question 4). Honest disclosure earns points and carries no penalty on its own; undisclosed use is an academic integrity violation under the course syllabus.

When you place the kit, your `assignments` folder will look like this:

```text
stat303-1/
├── .venv/
└── assignments/
    ├── assignment-a/
    ├── assignment-b/
    ├── assignment-c/
    ├── assignment-d/
    ├── assignment-e/
    └── assignment-f/         ← the extracted kit
        ├── assignment_f.ipynb
        ├── README.md
        └── data/
            ├── Housing_complete.csv
            ├── Housing_missing_price.csv
            └── house_features_and_price.csv
```

## Points Summary {-}

| Section | Points |
|---|---:|
| 1. Missing value imputation | 47 |
| 2. Binning | 33 |
| 3. Outliers | 15 |
| 4. AI Use Disclosure | 2 |
| 5. Formatting (instruction 8 above) | 3 |
| **Total** | **100** |
| Bonus: choosing $K$ by cross-validation (optional) | 10 |

## 1. Missing Value Imputation (47 points) {-}

`data/Housing_missing_price.csv` and `data/Housing_complete.csv` describe the same 545 houses: their area, bedrooms, bathrooms, and other features, and their price. The two files are identical except that some prices are missing in `Housing_missing_price.csv`. You will try several ways to impute the missing prices, then use `Housing_complete.csv` to check how close each method came, as [Data Cleaning](Data_cleaning.ipynb) did with GDP per capita.

You **cannot** use `Housing_complete.csv` to impute a price. Impute as if you did not have it, and use it only afterwards, to score the imputation. Each method starts from the original `housing_missing_price`: store its imputed prices in a new Series rather than overwriting the `price` column.

The starter supplies the code that reads the data:

```python
housing_missing_price = pd.read_csv("data/Housing_missing_price.csv")
housing_complete = pd.read_csv("data/Housing_complete.csv")
```

and a function, as in Data Cleaning, that plots imputed prices against the true prices and returns the RMSE. Use it in Questions 1.4, 1.6, and 1.7:

```python
def score_imputation(imputed_price, true_price, method):
    """Plot imputed prices against the true prices, and return the RMSE.

    Pass only the rows whose price was missing, for example
    filled_price.loc[index_null_price] and housing_complete.loc[index_null_price, "price"].
    """
    rmse = np.sqrt(((imputed_price - true_price) ** 2).mean())

    fig, ax = plt.subplots(figsize=(6, 4.5), layout="constrained")
    ax.scatter(true_price, imputed_price, s=50)
    ax.axline((0, 0), slope=1, color="gray", linestyle="--", label="perfect guess")
    ax.xaxis.set_major_formatter("${x:,.0f}")
    ax.yaxis.set_major_formatter("${x:,.0f}")
    ax.set(xlabel="True price", ylabel="Imputed price",
           title=f"{method}: RMSE = ${rmse:,.0f}")
    ax.legend()
    plt.show()
    return rmse
```

### 1.1 Number of missing values (2 points) {-}

How many values of `price` are missing in `housing_missing_price`?

### 1.2 Row labels of the missing values (3 points) {-}

Find the row labels where `price` is missing, and store them as a list or an array in `index_null_price`.

### 1.3 The most correlated numeric variable (2 points) {-}

Among the numeric columns, which variable has the highest correlation with `price`? Call it $A_{cont}$.

### 1.4 Imputation with a trendline (10 points) {-}

Make a scatterplot of `price` against $A_{cont}$ with a trendline. Fit the trendline with `np.polyfit()` on the rows where `price` is known, and use it to impute only the missing prices. Then call the supplied `score_imputation()` to plot the imputed prices against the true prices and report the RMSE, for the imputed rows only.

### 1.5 Dummy variables and the most correlated categorical variable (5 points) {-}

Encode the categorical columns of `housing_missing_price` as dummy variables, one 0/1 column per category, keeping the numeric columns as they are. Name the result `housing_dummy`. Which categorical variable is most highly correlated with `price`? Call it $V_{cat}$.

**Hint:** `pd.get_dummies()` with `dtype=int`, from [Data Preparation](Data_preparation.ipynb). Called on the whole DataFrame, it encodes every text column and leaves the numeric ones unchanged.

### 1.6 Imputation with a group mean (10 points) {-}

Impute each missing price as the mean price of the houses with the same value of $V_{cat}$. For example, if $V_{cat}$ were `basement`, a missing price for a house with a basement would be the mean price of all houses with a basement. Score the imputation with `score_imputation()`.

**Hint:** `groupby()` with `transform("mean")`, from [Data Cleaning](Data_cleaning.ipynb).

### 1.7 Imputation with a trendline within each group (15 points) {-}

Run the supplied code:

```python
sns.lmplot(data=housing_missing_price, x="area", y="price",
           hue="airconditioning", height=5, aspect=1.5)
plt.show()
```

It shows that the trendline of `price` against `area` differs between houses with and without air conditioning. Fit one trendline for each value of `airconditioning`, impute each missing price with the trendline that matches its house, and score the imputation with `score_imputation()`. Which of the methods in Questions 1.4, 1.6, and 1.7 imputed the prices most accurately?

**Hint:** Filter the known-price rows for each group with a Boolean mask, fit `np.polyfit()` on each, and fill the missing rows of each group with `.loc`.

## 2. Binning (33 points) {-}

`data/house_features_and_price.csv` holds 275 houses with their age, distance to the nearest MRT (metro) station, number of nearby convenience stores, location, and price. Binning a variable, as in [Data Preparation](Data_preparation.ipynb), can show how `house_price` changes with it in ways a single trendline cannot.

The starter supplies the code that reads the data:

```python
house_features = pd.read_csv("data/house_features_and_price.csv")
```

### 2.1 Trend with house age (4 points) {-}

Make a scatterplot of `house_price` against `house_age` with a trendline. What trend does the trendline show?

### 2.2 Equal-count bins of house age (3 points) {-}

Bin `house_age` into 5 bins that hold approximately the same number of houses.

### 2.3 Mean price by age bin (3 points) {-}

Plot the mean `house_price` for each `house_age` bin.

### 2.4 Comparing the two trends (4 points) {-}

Is the trend in Question 2.3 different from the trend in Question 2.1? If yes, is one of them incorrect? Explain.

### 2.5 The more informative trend (3 points) {-}

Is one of the two trends more informative? If yes, which one, and what does it show that the other does not?

### 2.6 Mean price by number of convenience stores (3 points) {-}

Make a bar plot of the mean `house_price` for each value of `number_convenience_stores`.

### 2.7 Bins of the number of convenience stores (8 points) {-}

Bin `number_convenience_stores` into as few bins as possible while keeping the non-linear trend of `house_price` you saw in Question 2.6. Then plot the mean `house_price` for each bin.

**Hint:** Choose the bin edges yourself and pass them to `pd.cut()` with `bins=`.

### 2.8 Sizes of the convenience-store bins (5 points) {-}

Print the number of houses in each bin from Question 2.7. Are the bins approximately equal in size? If not, is it reasonable to use bins of unequal sizes to show the trend of `house_price`? Explain.

## 3. Outliers (15 points) {-}

Continue with `house_features`.

### 3.1 Outlying prices (8 points) {-}

How many houses have an outlying `house_price` by the z-score method (|z| > 3), and how many by the IQR method (1.5 × IQR beyond the quartiles)? Do the two methods flag the same houses?

### 3.2 What makes them different (7 points) {-}

How do the houses flagged in Question 3.1 differ from the rest of the dataset? Explore their other columns, and state a hypothesis for why their prices are so extreme.

## 4. AI Use Disclosure (2 points) {-}

In the AI Use Disclosure section of your notebook, state whether **generative AI tools** were used to complete any part of this assignment.

- If **no AI was used**, write:
  *"I did not use generative AI tools on this assignment."*

- If **AI was used**, you must specify:
  1. **Tool(s)** used (e.g., ChatGPT, GitHub Copilot, Claude, etc.)
  2. **How** they were used (e.g., explaining a concept, interpreting an error message, debugging code you wrote)
  3. **Where** they influenced your work (which questions/sections)
  4. **Edits & verification** you made (how you checked correctness, what you modified)

### AI Use Template {-}

- **Used AI?** Yes / No
- **Tool(s):**
- **Purpose / How used (1–3 sentences):**
- **Scope (which questions/sections):**
- **Edits & verification (what you changed and how you checked correctness):**

> **Example (for illustration only):**
> - Used AI? Yes
> - Tools: ChatGPT (free)
> - How used: Asked why `groupby().transform()` returns as many rows as the DataFrame, and for help reading a `KeyError` in my bin labels.
> - Scope: 1.6, 2.7.
> - Edits & verification: Rewrote the code in my own style; checked that the imputed prices of the two air-conditioning groups matched the two group means.

**Grading (2 points):**
- **2 points** = a clear statement that you did not use AI, or a complete, specific and honest disclosure (tools named, usage and verification described)
- **1 point** = a disclosure that is vague or missing required details
- **0 points** = no statement at all, or a disclosure that is misleading

## Bonus: Choosing $K$ by Cross-Validation (10 points, optional) {-}

This is an optional question with no partial credit. You will get points only if your solution is completely correct. We advise you to attempt it only when you are done with the rest of the assignment. Loops are allowed.

In [Data Cleaning](Data_cleaning.ipynb), KNN imputation used a number of neighbors, $K$, picked by hand. Here you will choose $K$ from the data with **10-fold cross-validation**, then use it to impute the missing prices. You **cannot** use `Housing_complete.csv` to choose $K$.

(a) Keep the rows of `housing_dummy` whose `price` is known, and reset their row labels with `reset_index(drop=True)`, which renumbers the rows 0, 1, 2, … and discards the old labels; call the result `housing_known`. Rescale every column of `housing_known` to run from 0 to 1 with `MinMaxScaler`. Split the rows into 10 folds in order: the first tenth of the rows is fold 1, the next tenth fold 2, and so on. `np.array_split(np.arange(n), 10)` returns the row positions of each fold for `n` rows ([NumPy documentation](https://numpy.org/doc/stable/reference/generated/numpy.array_split.html)).

(b) For each $K$ from 1 to 50, and for each fold: set the fold's prices to missing, fill them with `KNNImputer(n_neighbors=K)`, convert the filled prices back to dollars with the scaler's `inverse_transform()`, and compute the RMSE against the fold's true prices. Average the 10 RMSEs to get the cross-validation RMSE for that $K$. Plot the cross-validation RMSE against $K$, and report the $K$ with the lowest value.

(c) Impute the missing prices of `housing_dummy` with KNN and that $K$, using the scaler fitted in (a): `fit_transform()` learns each column's range and rescales it, while `transform()` rescales with the range it already learned, so call `scaler.transform(housing_dummy)`. Then score the imputation with `score_imputation()`. Is it more accurate than the methods in Section 1?

## Prepare and Submit the HTML {-}

Restart the kernel, run all cells in order, resolve unexpected errors, and save. Render from the assignment folder. The terminal starts in `stat303-1`, so first move it there with `cd assignments` and then `cd assignment-f`, and check the current directory. Then run:

```text
quarto render assignment_f.ipynb --to html
```

Open `assignment_f.html` and check that your code, output, figures, and written answers are visible. Upload it to Assignment F in Canvas.

This step is graded through the formatting points in instruction 8.